In [1]:
# Run this only once if packages are missing:
# %pip install -q fasttext-wheel langdetect pandas
print('Using existing environment packages.')

Using existing environment packages.


## Language detection from lyrics (FastText)

Reads `lyrics.csv`, detects the language of each song using the **first 200 characters** of its lyrics
via FastText's pre-trained language identification model (`lid.176.ftz`),
and writes `lyrics_lang.csv` to the same directory.

Output: all original columns from `lyrics.csv` + a new `language` column.

In [2]:
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import fasttext
import fasttext.FastText

# Idempotent monkey-patch: fix np.array(copy=False) error in NumPy 2.x
if not hasattr(fasttext.FastText._FastText, '_orig_predict'):
    fasttext.FastText._FastText._orig_predict = fasttext.FastText._FastText.predict

    def _safe_predict(self, text, k=1, threshold=0.0, on_unicode_error='strict'):
        import numpy as np
        _old_array = np.array
        def _array_compat(*args, **kwargs):
            kwargs.pop('copy', None)
            return _old_array(*args, **kwargs)
        np.array = _array_compat
        try:
            return fasttext.FastText._FastText._orig_predict(self, text, k=k, threshold=threshold, on_unicode_error=on_unicode_error)
        finally:
            np.array = _old_array

    fasttext.FastText._FastText.predict = _safe_predict
    print('NumPy 2.x patch applied.')
else:
    print('Patch already applied, skipping.')

PROJECT_ROOT = Path.cwd().parent  # go up from notebooks to project root

# -- Download FastText lid model if not cached --
MODEL_URL = 'https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz'
MODEL_PATH = PROJECT_ROOT / 'models' / 'lid.176.ftz'
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists():
    print(f'Downloading FastText lid model -> {MODEL_PATH} ...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print('Done.')
else:
    print(f'Model already cached at {MODEL_PATH}')

ft_model = fasttext.load_model(str(MODEL_PATH))
print('FastText model loaded.')

NumPy 2.x patch applied.
Model already cached at d:\Users\Documents\GitHub\lyrics_analysis\models\lid.176.ftz
FastText model loaded.


In [3]:
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
LYRICS_IN  = PROCESSED_DIR / 'lyrics.csv'
LANG_OUT   = PROCESSED_DIR / 'lyrics_lang.csv'

# DATABRICKS PATH
# LYRICS_IN = '/Volumes/songs_db/default/storage/lyrics.csv'
# LANG_OUT = '/Volumes/songs_db/default/storage/lyrics_lang.csv'

df = pd.read_csv(LYRICS_IN)
print(f'Loaded {len(df)} songs from {LYRICS_IN}')

def make_snippet(lyrics: str, n_chars: int = 200) -> str:
    if not isinstance(lyrics, str) or not lyrics.strip():
        return ''
    return lyrics[:n_chars].replace('\n', ' ').strip()

snippets = df['lyrics'].map(make_snippet)
valid_mask = snippets.ne('')
langs = pd.Series('unknown', index=df.index, dtype='object')

# FastText supports list input; this is much faster than row-wise apply.
if valid_mask.any():
    labels, _scores = ft_model.predict(snippets[valid_mask].tolist(), k=1)
    langs.loc[valid_mask] = [lbls[0].replace('__label__', '') for lbls in labels]

lang_df = df.copy()
lang_df['original_lang'] = langs
lang_df.to_csv(LANG_OUT, index=False)

print(f'Saved {len(lang_df)} rows -> {LANG_OUT}')
print('\nLanguage distribution:')
print(lang_df['original_lang'].value_counts().to_string())
display(lang_df.head(10))

Loaded 694 songs from d:\Users\Documents\GitHub\lyrics_analysis\data\processed\lyrics.csv
Saved 694 rows -> d:\Users\Documents\GitHub\lyrics_analysis\data\processed\lyrics_lang.csv

Language distribution:
original_lang
es         276
en         181
ja         146
unknown     42
zh          15
ko          10
tr           3
pt           3
eo           2
sw           2
it           2
hr           1
de           1
ru           1
la           1
fr           1
he           1
ar           1
tl           1
vi           1
lmo          1
gd           1
id           1


,artist,title,spotify_uri,lyrics,original_lang
0,"Ryan Castro, Kapo, Gangsta",LA VILLA,2ZyrAym0sRLwt4PhGotHuI,"Kapo, Ryan Castro, Gangsta\nQué chimba, SOG\nT...",es
1,Bad Bunny,BAILE INoLVIDABLE,2lTm559tuIvatlT1u0JYG2,Pensaba que contigo iba a envejecer\nEn otra v...,es
2,"El Bogueto, Yung Beef",Cuando No Era Cantante,6N2iccqxRInhTLHc2Fu3W0,"Jajajajaja\nJaja, QueHicisteBella\n\nComo ante...",es
3,La T y La M,Soy Favela,3TfRpsYPQSXqqramSoWlNg,¡Ay!\n\nHe caminado tantas calles pero no he e...,es
4,Max Carra,UWAIE - versión cumbia,6UO9DhCUq4ZbQz7PXwelFV,NaN,unknown
5,"Lauta, Amigo de Artistas, Tote",Puñaladas,4AL4EamHEBKPpdcFRkYdXN,"Si tu espejo hablara, te diría esto\n\nEsa car...",es
6,Bad Bunny,DtMF,3sK8wGT43QFpWrvNQsrQya,"Eh, eh, eh, eh\n\nOtro sunset bonito que veo e...",es
7,"Roze Oficial, Max Carra, Valen, RAMKY EN LOS C...",Tu jardín con enanitos,6X8DTIJEgHUjZynuds0E2f,"Con este tema, te voy a hacer viajar en el tie...",es
8,Bad Bunny,VOY A LLeVARTE PA PR,59D4DOkspUbWyMmbAPQkxZ,"Acho, PR es otra cosa\nYo la conocí en Miami, ...",es
9,"El Bogueto, Anuel AA, Fuerza Regida, Yung Beef",Cuando No Era Cantante - Remix,2pbSCYzxrG0wa6qcj8IyiE,Uh-uh-uh\nHow pretty that ass loo-oo-ooks\nIn ...,en
